In [ ]:
import cv2
import numpy as np

# ============================================================
# CONFIG
# ============================================================
VIDEO_PATH = "/home/xilinx/input.mp4"  
RESIZE_H = 224
RESIZE_W = 224
Q15_SCALE = 32768  # 2^15

# ============================================================
# VIDEO PREPROCESSOR (Q1.15)
# ============================================================
class VideoPreprocessorQ15:
    def __init__(self, video_path):
        self.cap = cv2.VideoCapture(video_path)
        if not self.cap.isOpened():
            raise RuntimeError(f"Cannot open video: {video_path}")

        # FPS metadata (PS-only, not used further here)
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        if self.fps <= 0:
            self.fps = 30.0

        self.width  = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.frame_count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

        self.prev_frame = self._read_frame()

    def _read_frame(self):
        ret, frame = self.cap.read()
        if not ret:
            return None
        return frame

    def preprocess_pair(self, frame0, frame1):
        # Resize
        f0 = cv2.resize(frame0, (RESIZE_W, RESIZE_H))
        f1 = cv2.resize(frame1, (RESIZE_W, RESIZE_H))

        # Normalize to [0,1]
        f0 = f0.astype(np.float32) / 255.0
        f1 = f1.astype(np.float32) / 255.0

        # Shift to [-1,1]
        f0 = 2.0 * f0 - 1.0
        f1 = 2.0 * f1 - 1.0

        # Quantize to Q1.15
        f0 = np.clip(f0 * Q15_SCALE, -32768, 32767).astype(np.int16)
        f1 = np.clip(f1 * Q15_SCALE, -32768, 32767).astype(np.int16)

        # HWC → CHW
        f0 = f0.transpose(2, 0, 1)   # (3,H,W)
        f1 = f1.transpose(2, 0, 1)

        # Temporal stacking
        input_tensor = np.concatenate([f0, f1], axis=0)  # (6,H,W)

        return input_tensor

    def get_next_tensor(self):
        if self.prev_frame is None:
            return None, None, None

        curr_frame = self._read_frame()
        if curr_frame is None:
            return None, None, None

        input_tensor = self.preprocess_pair(self.prev_frame, curr_frame)

        frame0 = self.prev_frame
        frame1 = curr_frame
        self.prev_frame = curr_frame

        return input_tensor, frame0, frame1

    def release(self):
        self.cap.release()